In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
csv_path = '/content/drive/MyDrive/first_25000_rows.csv'


In [3]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA

In [17]:

df = pd.read_csv(csv_path)
df['ts_event'] = pd.to_datetime(df['ts_event'])
df = df.sort_values('ts_event').reset_index(drop=True)

def calc_ofi(now, prev, lvl):
    bpx_n, bpx_p = now[f'bid_px_{lvl}'], prev[f'bid_px_{lvl}']
    bsz_n, bsz_p = now[f'bid_sz_{lvl}'], prev[f'bid_sz_{lvl}']
    apx_n, apx_p = now[f'ask_px_{lvl}'], prev[f'ask_px_{lvl}']
    asz_n, asz_p = now[f'ask_sz_{lvl}'], prev[f'ask_sz_{lvl}']

    ofb = bsz_n - bsz_p if bpx_n == bpx_p else (bsz_n if bpx_n > bpx_p else -bsz_p)
    ofa = asz_n - asz_p if apx_n == apx_p else (-asz_n if apx_n > apx_p else asz_p)
    return ofb - ofa

def compute_all_ofis(df, lvl_ct=10):
    rows = []
    for sym in df['symbol'].unique():
        sub = df[df['symbol'] == sym].reset_index(drop=True)
        for i in range(1, len(sub)):
            now, prev = sub.iloc[i], sub.iloc[i-1]
            entry = {'ts_event': now['ts_event'], 'symbol': sym}
            for j in range(lvl_ct):
                lvl = f'{j:02d}'
                entry[f'ofi_{lvl}'] = calc_ofi(now, prev, lvl)
            rows.append(entry)
    return pd.DataFrame(rows)

def aggregate_ofis(ofi_df, lvl_ct=10):
    lvl_cols = [f'ofi_{i:02d}' for i in range(lvl_ct)]
    ofi_df['ofi_best'] = ofi_df['ofi_00']
    ofi_df['ofi_multi_sum'] = ofi_df[lvl_cols].sum(axis=1)

    pca_weights = (
        ofi_df.groupby('symbol')[lvl_cols]
        .apply(lambda x: PCA(1).fit(x).components_[0])
        .apply(lambda x: x / np.sum(np.abs(x)))
    )

    ofi_df['ofi_integrated'] = ofi_df.apply(
        lambda row: np.dot(row[lvl_cols], pca_weights[row['symbol']]), axis=1
    )
    return ofi_df

def resample_with_cross_asset(ofi_df):
    ofi_df['ts_event'] = pd.to_datetime(ofi_df['ts_event'])

    resampled_list = []
    for sym in ofi_df['symbol'].unique():
        temp = ofi_df[ofi_df['symbol'] == sym].set_index('ts_event')
        grouped = temp.resample('1min').sum().reset_index()
        grouped['symbol'] = sym
        resampled_list.append(grouped)

    resampled = pd.concat(resampled_list, ignore_index=True)


    cross_vals = []
    for ts, group in resampled.groupby('ts_event'):
        total = group['ofi_integrated'].sum()
        cross = total - group['ofi_integrated']
        cross_vals.extend(cross.values)

    resampled['cross_asset_ofi'] = cross_vals
    return resampled


ofi_raw = compute_all_ofis(df)
ofi_agg = aggregate_ofis(ofi_raw)
ofi_final = resample_with_cross_asset(ofi_agg)


ofi_final.head()

,ts_event,symbol,ofi_00,ofi_01,ofi_02,ofi_03,ofi_04,ofi_05,ofi_06,ofi_07,ofi_08,ofi_09,ofi_best,ofi_multi_sum,ofi_integrated,cross_asset_ofi
0,2024-10-21 11:54:00+00:00,AAPL,-195,600,10,-30,25,400,44,155,400,1,-195,1410,135.589451,0.0
1,2024-10-21 11:55:00+00:00,AAPL,-1115,781,1370,27,11,-84,-10,-110,-100,-55,-1115,715,138.624308,0.0
2,2024-10-21 11:56:00+00:00,AAPL,199,0,1200,0,0,0,0,1,0,0,199,1400,125.956942,0.0
3,2024-10-21 11:57:00+00:00,AAPL,500,592,50,4,-840,-118,-234,-88,-454,-346,500,-934,-152.295854,0.0
4,2024-10-21 11:58:00+00:00,AAPL,477,880,572,432,226,-412,256,88,544,291,477,3354,278.067413,0.0


In [19]:

output_path = '/content/drive/MyDrive/ofi_features_output.csv'
ofi_final.to_csv(output_path, index=False)

print(ofi_final.head())


                   ts_event symbol  ofi_00  ofi_01  ofi_02  ofi_03  ofi_04  \
0 2024-10-21 11:54:00+00:00   AAPL    -195     600      10     -30      25   
1 2024-10-21 11:55:00+00:00   AAPL   -1115     781    1370      27      11   
2 2024-10-21 11:56:00+00:00   AAPL     199       0    1200       0       0   
3 2024-10-21 11:57:00+00:00   AAPL     500     592      50       4    -840   
4 2024-10-21 11:58:00+00:00   AAPL     477     880     572     432     226   

   ofi_05  ofi_06  ofi_07  ofi_08  ofi_09  ofi_best  ofi_multi_sum  \
0     400      44     155     400       1      -195           1410   
1     -84     -10    -110    -100     -55     -1115            715   
2       0       0       1       0       0       199           1400   
3    -118    -234     -88    -454    -346       500           -934   
4    -412     256      88     544     291       477           3354   

   ofi_integrated  cross_asset_ofi  
0      135.589451              0.0  
1      138.624308              0.0  